<a href="https://colab.research.google.com/github/anuragN2107/lexguard-contract-reviewer/blob/main/lexguard_contract_reviewer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Project Name
**LexGuard AI:** High-Throughput Multi-Label Legal Contract Risk Identification & Triage Engine.


#Business Problem
Corporate legal teams, procurement divisions, and M&A operations handle thousands of dense, long-form PDF contracts (NDAs, MSAs, Vendor Agreements). Reviewing these documents manually is time-consuming, expensive, and error-prone. Missing a single unfavorable indemnity clause or an overly restrictive non-compete boundary can expose an enterprise to million-dollar lawsuits or regulatory fines.



#Objectives
* **Automate Clause Classification:** Instantly sort text segments into distinct legal categories (e.g., Indemnity, Non-Compete, Governing Law).
* **Flag Multi-Label Risks Simultaneously:** Detect when a single clause contains overlapping risks (e.g., a clause that is both a Limitation of Liability and a Termination Penalty).
* **Scale Processing Safely:** Handle large corporate documents asynchronously to prevent application timeouts or memory crashes.



# Data Overview
We utilize data inspired by the CUAD (Contract Understanding Atticus Dataset). It comprises expert-annotated contracts broken down into granular paragraphs or clauses. Each clause can map to zero, one, or multiple labels (e.g., Non-Compete, Indemnity, Governing Law, Limitation of Liability).

#Methodology
* **Chunking & Preprocessing:**Slide through raw contract text using overlapping token windows to capture context across artificial page limits.
* **Hybrid Embedding & Feature Extraction:** Pass tokens through a pre-trained nlpaueb/legal-bert-base-uncased transformer. Instead of relying solely on the static [CLS] token, feed the full contextual sequence hidden states into a multi-kernel 1D Convolutional Neural Network (TextCNN).
* **Multi-Label Head:** Pass max-pooled feature maps to a dense classification layer with a Sigmoid activation function to evaluate independent probability distributions per label.


#Why Choose a Legal-BERT + TextCNN Hybrid?
Standard Transformers can lose subtle n-gram specific phrasings if you compress the text down to a simple [CLS] pooler representation. TextCNN uses parallel convolutional filters ($3$, $4$, and $5$ tokens wide) to capture key local phrases like "shall not compete within a radius of" or "indemnify and hold harmless" regardless of where they sit in a long sentence. This cuts down inference latency significantly compared to fully fine-tuning giant generative LLMs, making it ideal for high-throughput enterprise pipelines.


#Tools & Requirements
* **Hardware / Environment:** Google Colab (T4 GPU runtime), Docker Desktop, Hugging Face Hub Account.
* **Core Libraries:** torch, transformers, datasets, fastapi, celery, redis, uvicorn, pydantic.

#Step 1: Data Synthesis & Setup
To install core dependencies and generate a synthetic multi-label dataset modeled after CUAD for testing.

In [ ]:
# Install required libraries
!pip install -q transformers datasets torch scikit-learn pandas

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score


In [ ]:
# 1. Create Multi-Label Synthetic Legal Clause Dataset
data = {
    "text": [
        "The Vendor agrees to defend, indemnify, and hold harmless the Client from all claims, damages, liabilities, and expenses arising from third-party IP infringement.",
        "During the term of this Agreement and for a period of two (2) years thereafter, the Executive shall not directly or indirectly engage in any business competing with the Company.",
        "This Agreement shall be governed by, and construed in accordance with, the domestic laws of the State of Delaware without giving effect to any choice of law provision.",
        "In no event shall either party's aggregate liability under this agreement exceed the total fees paid by the client in the twelve (12) months preceding the claim event.",
        "Except as otherwise provided herein, either party may terminate this agreement without cause upon providing ninety (90) days prior written notice to the other party.",
        "The Executive shall not solicit, induce, or attempt to induce any employee or independent contractor of the Company to terminate their relationship with the Company."
    ],
    # Labels: [Indemnity, Non-Compete, Governing_Law, Liability_Cap]
    "labels": [
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1],
        [0, 0, 0, 0],
        [0, 1, 0, 0]
    ]
}

df = pd.DataFrame(data)
df = pd.concat([df] * 10, ignore_index=True)
print(f"Dataset generated. Total rows: {len(df)}")

Dataset generated. Total rows: 60


#Step 2: Custom PyTorch Dataset

In [ ]:
class LegalContractDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.text = dataframe['text'].values
        self.labels = dataframe['labels'].values
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        text = str(self.text[index])
        inputs = self.tokenizer.encode_plus(
            text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_token_type_ids=False,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[index], dtype=torch.float)
        }

# Initialize Tokenizer using Legal-BERT
TOKENIZER_NAME = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = LegalContractDataset(train_df, tokenizer)
val_dataset = LegalContractDataset(val_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

#Step 3: Architecture Definition (Legal-BERT + TextCNN)
The full contextualized sequence outputs from Legal-BERT are passed into parallel 1D Convolutional layers with variable kernel sizes ($3, 4, 5$).

In [ ]:
class LegalBertCNNClassifier(nn.Module):
    def __init__(self, bert_model_name, num_classes, num_filters=100, filter_sizes=[3, 4, 5]):
        super(LegalBertCNNClassifier, self).__init__()
        # Freeze or unfreeze BERT backbones based on compute limits
        self.bert = AutoModel.from_pretrained(bert_model_name)
        embedding_dim = self.bert.config.hidden_size # 768 for base-uncased

        # 1D Convolutional layers for different n-gram extractions
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])

        # Classification layer
        self.fc = nn.Linear(len(filter_sizes) * num_filters, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, input_ids, attention_mask):
        # 1. Extract BERT hidden states
        bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # last_hidden_state shape: [batch_size, sequence_length, embedding_dim]
        hidden_states = bert_outputs.last_hidden_state

        # Permute for PyTorch 1D Conv input format: [batch_size, embedding_dim, sequence_length]
        hidden_states = hidden_states.permute(0, 2, 1)

        # 2. Extract multi-gram patterns using TextCNN structure
        pooled_outputs = []
        for conv in self.convs:
            x = torch.relu(conv(hidden_states))
            # Global Max Pooling over the sequence dimension
            x = torch.max(x, dim=2)[0]
            pooled_outputs.append(x)

        # Concatenate features from all kernels
        cat = self.dropout(torch.cat(pooled_outputs, dim=1))

        # 3. Dense layer to compute logits
        logits = self.fc(cat)
        return logits

# Instantiate Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LegalBertCNNClassifier(TOKENIZER_NAME, num_classes=4)
model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LegalBertCNNClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elemen

# Step 4: Training & Multi-Label Risk Evaluation Loop
Use BCE With LogitsLoss for this task. It applies an independent sigmoid function over every output neuron, treating each legal risk calculation as a standalone probability test.

In [ ]:
class LegalContractDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe.reset_index(drop=True) # Reset index to avoid mapping errors
        self.tokenizer = tokenizer
        self.text = self.df['text'].values
        # Convert lists to a clean 2D numpy array
        self.labels = np.array(self.df['labels'].tolist(), dtype=np.float32)
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        text = str(self.text[index])
        inputs = self.tokenizer.encode_plus(
            text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_token_type_ids=False,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[index], dtype=torch.float)
        }

# Re-initialize the loaders safely
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = LegalContractDataset(train_df, tokenizer)
val_dataset = LegalContractDataset(val_df, tokenizer)

# Important: Keep batch size small for testing
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

#Step 5: Evaluate the Model
Before deploying,must check how well the model handles unseen data. Because this is a multi-label task, we use a threshold (usually $0.5$) to turn the model's raw probability scores into $0$ or $1$ predictions, then calculate the Macro F1-Score.

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import transformers # Explicitly import the root namespace
from sklearn.metrics import classification_report

print("1. Forcing clean tokenizer instantiation directly from the root module...")
absolute_clean_tokenizer = transformers.AutoTokenizer.from_pretrained(
    "nlpaueb/legal-bert-base-uncased",
    use_fast=True
)

# 2. Dataset using the direct call method instead of .encode_plus
class FinalBulletproofDataset(Dataset):
    def __init__(self, dataframe, tokenizer_obj, max_len=128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer_obj
        self.text = self.df['text'].values
        self.labels = np.array(self.df['labels'].tolist(), dtype=np.float32)
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        text_sample = str(self.text[index])

        inputs = self.tokenizer(
            text_sample,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_token_type_ids=False,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[index], dtype=torch.float)
        }

print("2. Constructing safe validation data infrastructure...")
final_eval_dataset = FinalBulletproofDataset(val_df, absolute_clean_tokenizer, max_len=128)

# Setting num_workers=0 avoids background thread variable pollution issues in Colab
final_val_loader = DataLoader(final_eval_dataset, batch_size=4, shuffle=False, num_workers=0)

# 3. Processing loop execution
model.eval()
all_preds = []
all_labels = []

print("3. Executing evaluation loop tracking matrix conversions...")
with torch.no_grad():
    for batch in final_val_loader:
        # Extract pure data matrices safely
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].numpy()

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs > 0.5).astype(int)

        all_preds.append(preds)
        all_labels.append(labels)

# 4. Consolidate results across iterations
all_preds = np.vstack(all_preds)
all_labels = np.vstack(all_labels)

target_names = ["Indemnity", "Non-Compete", "Governing Law", "Liability Cap"]
print("\n=== Validation Performance Report ===")
print(classification_report(all_labels, all_preds, target_names=target_names, zero_division=0))

1. Forcing clean tokenizer instantiation directly from the root module...
2. Constructing safe validation data infrastructure...
3. Executing evaluation loop tracking matrix conversions...

=== Validation Performance Report ===
               precision    recall  f1-score   support

    Indemnity       0.00      0.00      0.00         5
  Non-Compete       0.17      1.00      0.29         2
Governing Law       0.00      0.00      0.00         1
Liability Cap       0.25      1.00      0.40         3

    micro avg       0.21      0.45      0.29        11
    macro avg       0.10      0.50      0.17        11
 weighted avg       0.10      0.45      0.16        11
  samples avg       0.21      0.42      0.28        11



In [ ]:
import os
import torch
from google.colab import files

# 1. Force save from live memory to a guaranteed location
print("Saving model weights from live memory...")
torch.save(model.state_dict(), "/content/legal_bert_cnn.pt")

# 2. Double check that the file physically exists now
if os.path.exists("/content/legal_bert_cnn.pt"):
    print("Success! File safely created. Triggering download window now...")
    files.download("/content/legal_bert_cnn.pt")
else:
    print("Error: The model object isn't in your current workspace memory. You need to re-run your Step 4 training loop cell first.")

Saving model weights from live memory...
Success! File safely created. Triggering download window now...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Performance Analysis & Optimization Insights
* **Mitigating Overfitting:** Multi-label target spaces are highly sparse; most clauses don't contain most risks. Standard cross-entropy can result in models predicting zero for everything. Using BCEWithLogitsLoss prevents this, but you should also track the Macro F1-Score rather than global accuracy to ensure rare risk terms aren't ignored.

* **The Power of the Hybrid Setup:** Passing tokens to TextCNN layers cuts processing times relative to long-context attention models, reducing pipeline operating costs when working with massive document batches.



# Future Improvements
* **Transition to Hierarchical Attention Networks (HAN):** Upgrading the text aggregation step to a Hierarchical Attention setup would let the system flag structural risks at both the individual word level and the broader document level simultaneously.

* **Add Quantization (ONNX / TensorRT)**:Converting the PyTorch weights to an 8-bit quantized format runtime allows workers to process files much faster while using a fraction of the CPU memory.